# project_schema_and_plan



- Project objective and supported tasks
- Form 1065 and Form 1120 scope
- Supported tax years
- Base-model requirements
- Dataset folder structure
- Corpus and metadata schemas
- Training, validation, and test strategy
- Evaluation metrics
- Complete LLM development workflow
- File naming and versioning rules

# Data Cleaning and Preprocessing

## Section 1: Kaggle Environment Setup

This section checks:

- Python version
- Kaggle environment
- GPU availability
- Memory and storage
- Random seed

A GPU is optional for data cleaning but may help with OCR.

In [1]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import psutil

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# Kaggle environment
IS_KAGGLE = Path("/kaggle").exists()
INPUT_DIR = Path("/kaggle/input") if IS_KAGGLE else Path("input")
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python version : {sys.version.split()[0]}")
print(f"Kaggle detected: {IS_KAGGLE}")
print(f"Input folder   : {INPUT_DIR}")
print(f"Output folder  : {OUTPUT_DIR}")
print(f"Random seed    : {RANDOM_SEED}")

Python version : 3.12.13
Kaggle detected: True
Input folder   : /kaggle/input
Output folder  : /kaggle/working
Random seed    : 42


In [2]:
# GPU check
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total",
         "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"GPU             : {result.stdout.strip()}")
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU             : No NVIDIA GPU detected")

GPU             : Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
# Memory and storage check
GB = 1024 ** 3

memory = psutil.virtual_memory()
disk = shutil.disk_usage(OUTPUT_DIR)

print(f"Total memory    : {memory.total / GB:.2f} GB")
print(f"Available memory: {memory.available / GB:.2f} GB")
print(f"Free storage    : {disk.free / GB:.2f} GB")

Total memory    : 31.35 GB
Available memory: 30.21 GB
Free storage    : 19.50 GB


## Section 2: Install and Import Required Libraries

This section prepares the libraries needed for PDF extraction, OCR, text cleaning, duplicate detection, and dataset export.

In [4]:
%pip install -q pypdf pymupdf pdfplumber pytesseract rapidfuzz beautifulsoup4 lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 88.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.
Note: you may need t

In [5]:
import re
import json
import hashlib
import shutil
from pathlib import Path

import fitz
import pandas as pd
import pdfplumber
import pytesseract

from bs4 import BeautifulSoup
from PIL import Image
from pypdf import PdfReader
from rapidfuzz import fuzz
from tqdm.auto import tqdm

print("All required Python libraries imported successfully.")

All required Python libraries imported successfully.


## Section 3: Configure Dataset Paths

This section locates the source documents and creates folders for processed outputs.

In [6]:
# Display attached Kaggle datasets
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/tax_llm")

dataset_folders = [path for path in INPUT_DIR.iterdir() if path.is_dir()]

print("Attached Kaggle datasets:")
for folder in dataset_folders:
    print(f"- {folder.name}")

Attached Kaggle datasets:
- datasets


In [7]:
# Create output folders
CLEANED_DIR = OUTPUT_DIR / "cleaned_documents"
REPORTS_DIR = OUTPUT_DIR / "reports"
EXPORT_DIR = OUTPUT_DIR / "exports"

for folder in [OUTPUT_DIR, CLEANED_DIR, REPORTS_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Input folder   : {INPUT_DIR}")
print(f"Output folder  : {OUTPUT_DIR}")
print(f"Cleaned folder : {CLEANED_DIR}")
print(f"Reports folder : {REPORTS_DIR}")
print(f"Export folder  : {EXPORT_DIR}")

Input folder   : /kaggle/input
Output folder  : /kaggle/working/tax_llm
Cleaned folder : /kaggle/working/tax_llm/cleaned_documents
Reports folder : /kaggle/working/tax_llm/reports
Export folder  : /kaggle/working/tax_llm/exports


In [8]:
SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

source_files = [
    path
    for path in INPUT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
]

print(f"Supported source files found: {len(source_files)}")

for path in source_files[:20]:
    print(path)

Supported source files found: 58
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065x--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/publications/p541--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065x--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sd--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sb2--2018.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065s23--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data

## Section 3: Configure Dataset Paths

This section locates the raw tax documents and creates folders for processed outputs.

In [9]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

dataset_folders = sorted(
    path for path in INPUT_DIR.iterdir() if path.is_dir()
)

print(f"Attached datasets: {len(dataset_folders)}")

for index, folder in enumerate(dataset_folders):
    print(f"{index}: {folder}")

Attached datasets: 1
0: /kaggle/input/datasets


In [10]:
if len(dataset_folders) != 1:
    raise ValueError(
        "Expected one attached dataset. Select the correct folder manually."
    )

RAW_DATA_DIR = dataset_folders[0]

print(f"Raw data folder: {RAW_DATA_DIR}")

Raw data folder: /kaggle/input/datasets


## Create output folders

In [11]:
OUTPUT_DIR = Path("/kaggle/working/tax_llm")

CLEANED_DIR = OUTPUT_DIR / "cleaned_documents"
REPORTS_DIR = OUTPUT_DIR / "reports"
EXPORTS_DIR = OUTPUT_DIR / "exports"

for folder in [CLEANED_DIR, REPORTS_DIR, EXPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Cleaned documents: {CLEANED_DIR}")
print(f"Reports          : {REPORTS_DIR}")
print(f"Exports          : {EXPORTS_DIR}")

Cleaned documents: /kaggle/working/tax_llm/cleaned_documents
Reports          : /kaggle/working/tax_llm/reports
Exports          : /kaggle/working/tax_llm/exports


## Section 4: Discover Source Files

This section searches the selected dataset folder for supported tax-document files.

In [12]:
SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

source_files = sorted(
    path
    for path in RAW_DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f"Source files found: {len(source_files)}")

if source_files:
    for path in source_files[:20]:
        print(f"- {path.relative_to(RAW_DATA_DIR)}")

    if len(source_files) > 20:
        print(f"... and {len(source_files) - 20} more files")
else:
    print("No supported files found. Check RAW_DATA_DIR and the dataset contents.")

Source files found: 58
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065x--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/guides/p4163.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/guides/p4164.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065x--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/publications/p541--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065s23--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sb2--2018.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sd--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedul

## Section 5: Create the Document Inventory

This section creates a table containing basic information about every discovered source file.

In [13]:
import hashlib
import mimetypes

import pandas as pd
from tqdm.auto import tqdm


inventory_records = []

for file_path in tqdm(source_files, desc="Creating inventory"):
    relative_path = file_path.relative_to(RAW_DATA_DIR)
    file_stats = file_path.stat()

    # Create a stable ID from the relative file path
    document_id = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:16]

    inventory_records.append(
        {
            "document_id": document_id,
            "file_name": file_path.name,
            "relative_path": str(relative_path),
            "file_extension": file_path.suffix.lower(),
            "mime_type": mimetypes.guess_type(file_path.name)[0],
            "file_size_bytes": file_stats.st_size,
            "file_size_mb": round(file_stats.st_size / (1024 ** 2), 3),
        }
    )

document_inventory = pd.DataFrame(inventory_records)

print(f"Documents inventoried: {len(document_inventory)}")
display(document_inventory.head())

Creating inventory:   0%|          | 0/58 [00:00<?, ?it/s]

Documents inventoried: 58


,document_id,file_name,relative_path,file_extension,mime_type,file_size_bytes,file_size_mb
0,d304db736face09f,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,334608,0.319
1,afade3aec3a942e0,f1065x--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,168645,0.161
2,d1956ddbb121d7f5,p4163.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,1415699,1.350
3,caa77758d1eecb37,p4164.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,6909012,6.589
4,e54829d5a97de257,i1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,820966,0.783


## Review the inventory

In [14]:
print("Files by extension:")
display(
    document_inventory["file_extension"]
    .value_counts()
    .rename_axis("file_extension")
    .reset_index(name="file_count")
)

print(f"Total dataset size: {document_inventory['file_size_mb'].sum():.2f} MB")
print(f"Empty files found : {(document_inventory['file_size_bytes'] == 0).sum()}")

Files by extension:


,file_extension,file_count
0,.pdf,56
1,.csv,2


Total dataset size: 29.74 MB
Empty files found : 0


## save the inventory

In [15]:
inventory_path = REPORTS_DIR / "document_inventory.csv"

document_inventory.to_csv(inventory_path, index=False)

print(f"Inventory saved: {inventory_path}")

Inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 6: Detect Form 1065 and Form 1120 Documents

This section identifies whether each document relates to Form 1065, Form 1120, both forms, or neither form.

In [16]:
def get_identification_text(file_path, max_pdf_pages=3):
    """Read a small amount of text for form detection."""

    extension = file_path.suffix.lower()

    try:
        if extension == ".pdf":
            document = fitz.open(file_path)

            text = " ".join(
                document[page_number].get_text()
                for page_number in range(min(max_pdf_pages, len(document)))
            )

            document.close()
            return text

        if extension in {".txt", ".csv", ".json", ".jsonl"}:
            return file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )[:50_000]

        if extension in {".html", ".htm"}:
            html = file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            return BeautifulSoup(html, "lxml").get_text(" ")[:50_000]

    except Exception:
        return ""

    return ""

## Detect the return type

In [17]:
FORM_1065_PATTERN = re.compile(
    r"\bform[\s_-]*1065(?!\d)",
    re.IGNORECASE
)

FORM_1120_PATTERN = re.compile(
    r"\bform[\s_-]*1120(?![\s_-]?[a-z]|\d)",
    re.IGNORECASE
)


def detect_return_type(file_path):
    relative_path = str(
        file_path.relative_to(RAW_DATA_DIR)
    ).replace("_", " ").replace("-", " ")

    document_text = get_identification_text(file_path)

    searchable_text = f"{relative_path} {document_text[:50_000]}"

    has_1065 = bool(FORM_1065_PATTERN.search(searchable_text))
    has_1120 = bool(FORM_1120_PATTERN.search(searchable_text))

    if has_1065 and has_1120:
        return "both"
    elif has_1065:
        return "1065"
    elif has_1120:
        return "1120"
    else:
        return "unknown"

## Update the inventory

In [18]:
file_lookup = {
    str(path.relative_to(RAW_DATA_DIR)): path
    for path in source_files
}

detected_return_types = []

for relative_path in tqdm(
    document_inventory["relative_path"],
    desc="Detecting return types"
):
    file_path = file_lookup[relative_path]
    detected_return_types.append(
        detect_return_type(file_path)
    )

document_inventory["return_type"] = detected_return_types

display(
    document_inventory[
        ["file_name", "file_extension", "return_type"]
    ].head(20)
)

Detecting return types:   0%|          | 0/58 [00:00<?, ?it/s]

,file_name,file_extension,return_type
0,f1065--2025.pdf,.pdf,1065
1,f1065x--2025.pdf,.pdf,1065
2,p4163.pdf,.pdf,unknown
3,p4164.pdf,.pdf,unknown
4,i1065--2025.pdf,.pdf,1065
5,i1065x--2025.pdf,.pdf,1065
6,p541--2025.pdf,.pdf,1065
7,i1065s23--2025.pdf,.pdf,1065
8,i1065sb2--2018.pdf,.pdf,1065
9,i1065sd--2025.pdf,.pdf,1065


## Review and save results

In [19]:
return_type_summary = (
    document_inventory["return_type"]
    .value_counts(dropna=False)
    .rename_axis("return_type")
    .reset_index(name="document_count")
)

display(return_type_summary)

unknown_count = (
    document_inventory["return_type"] == "unknown"
).sum()

both_count = (
    document_inventory["return_type"] == "both"
).sum()

print(f"Unknown documents     : {unknown_count}")
print(f"Documents mentioning both forms: {both_count}")

inventory_path = REPORTS_DIR / "document_inventory.csv"
document_inventory.to_csv(inventory_path, index=False)

print(f"Updated inventory saved: {inventory_path}")

,return_type,document_count
0,1120,23
1,1065,19
2,unknown,12
3,both,4


Unknown documents     : 12
Documents mentioning both forms: 4
Updated inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 7: Detect the Tax Year

This section detects the tax year using the file name and the first pages of each document.

## Tax year Detrection Function

In [20]:
from collections import Counter
from datetime import datetime


MIN_TAX_YEAR = 1990
MAX_TAX_YEAR = datetime.now().year + 1

YEAR_PATTERN = rf"\b(?:{MIN_TAX_YEAR}|19[9][0-9]|20[0-9]{{2}})\b"


def detect_tax_year(file_path):
    file_name_text = file_path.stem.replace("_", " ").replace("-", " ")
    document_text = get_identification_text(file_path, max_pdf_pages=3)

    year_scores = Counter()
    year_sources = {}

    # Years found in the file name receive higher priority
    filename_years = re.findall(YEAR_PATTERN, file_name_text)

    for year in filename_years:
        year_number = int(year)

        if MIN_TAX_YEAR <= year_number <= MAX_TAX_YEAR:
            year_scores[year_number] += 5
            year_sources.setdefault(year_number, set()).add("file_name")

    # Search for clear tax-year statements
    tax_year_patterns = [
        rf"\btax\s+year\s+({YEAR_PATTERN})",
        rf"\bcalendar\s+year\s+({YEAR_PATTERN})",
        rf"\b({YEAR_PATTERN})\s+instructions?\s+for\s+form\b",
        rf"\b({YEAR_PATTERN})\s+form\s+(?:1065|1120)\b",
        rf"\bform\s+(?:1065|1120)\s*\(?({YEAR_PATTERN})\)?",
    ]

    for pattern in tax_year_patterns:
        for match in re.findall(pattern, document_text, flags=re.IGNORECASE):
            # Nested regex groups may return tuples
            if isinstance(match, tuple):
                match = next(
                    value for value in match
                    if value and re.fullmatch(r"\d{4}", value)
                )

            year_number = int(match)

            if MIN_TAX_YEAR <= year_number <= MAX_TAX_YEAR:
                year_scores[year_number] += 3
                year_sources.setdefault(year_number, set()).add("document_text")

    if not year_scores:
        return {
            "tax_year": None,
            "tax_year_source": "not_detected",
            "tax_year_status": "review",
        }

    ranked_years = year_scores.most_common()
    best_year, best_score = ranked_years[0]

    # Flag equal scores for manual review
    tied_years = [
        year for year, score in ranked_years
        if score == best_score
    ]

    status = "detected" if len(tied_years) == 1 else "ambiguous"

    return {
        "tax_year": best_year,
        "tax_year_source": ", ".join(
            sorted(year_sources.get(best_year, {"unknown"}))
        ),
        "tax_year_status": status,
    }

In [21]:
# Update the inventory

tax_year_results = []

for relative_path in tqdm(
    document_inventory["relative_path"],
    desc="Detecting tax years"
):
    file_path = file_lookup[relative_path]
    tax_year_results.append(detect_tax_year(file_path))


tax_year_table = pd.DataFrame(tax_year_results)

document_inventory["tax_year"] = pd.array(
    tax_year_table["tax_year"],
    dtype="Int64"
)

document_inventory["tax_year_source"] = (
    tax_year_table["tax_year_source"]
)

document_inventory["tax_year_status"] = (
    tax_year_table["tax_year_status"]
)

display(
    document_inventory[
        [
            "file_name",
            "return_type",
            "tax_year",
            "tax_year_source",
            "tax_year_status",
        ]
    ].head(20)
)

Detecting tax years:   0%|          | 0/58 [00:00<?, ?it/s]

,file_name,return_type,tax_year,tax_year_source,tax_year_status
0,f1065--2025.pdf,1065,2025,"document_text, file_name",detected
1,f1065x--2025.pdf,1065,2025,file_name,detected
2,p4163.pdf,unknown,<NA>,not_detected,review
3,p4164.pdf,unknown,<NA>,not_detected,review
4,i1065--2025.pdf,1065,2025,"document_text, file_name",detected
5,i1065x--2025.pdf,1065,2025,file_name,detected
6,p541--2025.pdf,1065,2025,file_name,detected
7,i1065s23--2025.pdf,1065,2025,"document_text, file_name",detected
8,i1065sb2--2018.pdf,1065,2018,file_name,detected
9,i1065sd--2025.pdf,1065,2025,file_name,detected


In [22]:
# Review and save the results

tax_year_summary = (
    document_inventory
    .groupby(["tax_year", "return_type"], dropna=False)
    .size()
    .reset_index(name="document_count")
    .sort_values(["tax_year", "return_type"])
)

display(tax_year_summary)

review_documents = document_inventory[
    document_inventory["tax_year_status"] != "detected"
]

print(f"Detected tax years : {(document_inventory['tax_year_status'] == 'detected').sum()}")
print(f"Needs review       : {len(review_documents)}")

if not review_documents.empty:
    display(
        review_documents[
            ["file_name", "return_type", "tax_year", "tax_year_status"]
        ].head(20)
    )

inventory_path = REPORTS_DIR / "document_inventory.csv"
document_inventory.to_csv(inventory_path, index=False)

print(f"Updated inventory saved: {inventory_path}")

,tax_year,return_type,document_count
0,2011,1120,2
1,2014,1065,1
2,2015,1120,1
3,2016,1120,2
4,2016,unknown,1
5,2018,1065,2
6,2018,1120,4
7,2018,both,1
8,2019,1065,1
9,2019,1120,1


Detected tax years : 54
Needs review       : 4


,file_name,return_type,tax_year,tax_year_status
2,p4163.pdf,unknown,<NA>,review
3,p4164.pdf,unknown,<NA>,review
38,p4163.pdf,unknown,<NA>,review
39,p4164.pdf,unknown,<NA>,review


Updated inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 8: Extract Text from Digital PDFs

This section extracts text from searchable PDF files and identifies PDFs that may require OCR.